<a href="https://colab.research.google.com/github/Shanmuganathan75/QM640-WALSH-CAPSTONE/blob/main/%5B06%5D-Statistical_Tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QM 640 Capstone — Step 6: RQ1-RQ4 Statistical Tests

Runs the exact tests from Table 1 of the Synopsis, each preceded by a
Shapiro-Wilk normality check with the specified non-parametric fallback.

- **RQ1:** One-sample t-test on CAR_short (H0: CAR = 0) — fallback: Wilcoxon signed-rank
- **RQ2:** One-way ANOVA across announcement_type — fallback: Kruskal-Wallis; Tukey HSD post-hoc if significant
- **RQ3:** Multiple regression with interaction (firm_size x announcement_type)
- **RQ4:** Independent-samples t-test, tech vs. non-tech — Levene's test for variance equality; Welch's correction if violated

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [ ]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 757, done.
remote: Counting objects: 100% (529/529), done.
remote: Compressing objects: 100% (319/319), done.
remote: Total 757 (delta 225), reused 503 (delta 209), pack-reused 228 (from 1)
Receiving objects: 100% (757/757), 5.02 MiB | 19.02 MiB/s, done.
Resolving deltas: 100% (322/322), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [ ]:
!pip install -q pandas numpy scipy statsmodels scikit-learn

## Cell 3 — Configuration

In [ ]:
import os

DATA_FILE = os.path.join(BASE_DIR, "data/processed/analysis_dataset.csv")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

ALPHA = 0.05

import pandas as pd
df = pd.read_csv(DATA_FILE)
print(f"Loaded {len(df)} events from {DATA_FILE}")
df.head()

Loaded 509 events from /content/QM640-WALSH-CAPSTONE/data/processed/analysis_dataset.csv


,event_id,ticker,event_date,announcement_type,firm_size_log,sector,alpha,beta,CAR_short,CAR_long,sector_binary
0,0001393883-25-000113:a2025-11dhisharebuybackpr...,DHX,2025-11-06,M&A,NaN,NaN,0.007345,0.328199,-0.060663,-0.378834,Non-Technology
1,0001213900-25-073334:ea025242001ex99-1_ondas.htm,ONDS,2025-08-08,M&A,NaN,NaN,-0.002127,1.895226,0.020943,0.738625,Non-Technology
2,0001493152-24-015849:ex99-1.htm,AITX,2024-04-25,R&D,NaN,NaN,-0.002079,1.233456,0.339011,0.864108,Non-Technology
3,0001213900-26-069795:ea029521601ex99-1.htm,MYSE,2026-06-18,partnership,NaN,NaN,0.003747,2.119101,0.224378,0.080728,Non-Technology
4,0001193125-26-278557:lmfa-ex99_1.htm,LMFA,2026-06-23,R&D,NaN,NaN,-0.012485,2.919025,0.041741,-0.289997,Non-Technology


## RQ1 — Does CAR differ significantly from zero?

In [ ]:
from scipy import stats

def rq1_test(df):
    car = df["CAR_short"].dropna()
    _, p_norm = stats.shapiro(car)
    print(f"Shapiro-Wilk normality test: p = {p_norm:.4f}")

    if p_norm >= ALPHA:
        t_stat, p_val = stats.ttest_1samp(car, 0)
        print(f"One-sample t-test: t = {t_stat:.3f}, p = {p_val:.4f}")
        method = "one-sample t-test"
    else:
        stat, p_val = stats.wilcoxon(car)
        t_stat = stat
        print(f"Normality violated -> Wilcoxon signed-rank test: W = {stat:.3f}, p = {p_val:.4f}")
        method = "Wilcoxon signed-rank (non-parametric fallback)"

    caar = car.mean()
    print(f"CAAR (mean CAR): {caar:.4%}")
    print(f"Result: {'REJECT' if p_val < ALPHA else 'FAIL TO REJECT'} H0 at alpha = .05")

    return {"RQ": "RQ1", "method": method, "statistic": t_stat, "p_value": p_val,
            "n": len(car), "CAAR": caar}


rq1_result = rq1_test(df)

Shapiro-Wilk normality test: p = 0.0000
Normality violated -> Wilcoxon signed-rank test: W = 54297.000, p = 0.0014
CAAR (mean CAR): -2.4583%
Result: REJECT H0 at alpha = .05


## RQ2 — Does CAR differ by announcement type?

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

def rq2_test(df):
    groups = [g["CAR_short"].dropna() for _, g in df.groupby("announcement_type")]

    normal = all(stats.shapiro(g)[1] >= ALPHA for g in groups if len(g) >= 3)
    _, p_levene = stats.levene(*groups)
    print(f"Normality (all groups): {'OK' if normal else 'violated'} | "
          f"Levene's test (equal variance): p = {p_levene:.4f}")

    if normal:
        f_stat, p_val = stats.f_oneway(*groups)
        print(f"One-way ANOVA: F = {f_stat:.3f}, p = {p_val:.4f}")
        method = "one-way ANOVA"
        if p_val < ALPHA:
            tukey = pairwise_tukeyhsd(df["CAR_short"].dropna(),
                                       df.loc[df["CAR_short"].notna(), "announcement_type"])
            print("\nTukey HSD post-hoc comparisons:")
            print(tukey)
    else:
        h_stat, p_val = stats.kruskal(*groups)
        f_stat = h_stat
        print(f"Normality violated -> Kruskal-Wallis: H = {h_stat:.3f}, p = {p_val:.4f}")
        method = "Kruskal-Wallis (non-parametric fallback)"

    ss_between = sum(len(g) * (g.mean() - df["CAR_short"].mean()) ** 2 for g in groups)
    ss_total = ((df["CAR_short"].dropna() - df["CAR_short"].mean()) ** 2).sum()
    eta_sq = ss_between / ss_total if ss_total else float("nan")
    print(f"Effect size (eta-squared): {eta_sq:.4f}")

    return {"RQ": "RQ2", "method": method, "statistic": f_stat, "p_value": p_val,
            "n": len(df), "eta_squared": eta_sq}


rq2_result = rq2_test(df)

Normality (all groups): violated | Levene's test (equal variance): p = 0.0613
Normality violated -> Kruskal-Wallis: H = 3.878, p = 0.1439
Effect size (eta-squared): 0.0067


## RQ3 — Does firm size moderate announcement-type -> CAR?

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

def rq3_test(df):
    model_df = df.dropna(subset=["CAR_short", "firm_size_log", "announcement_type"]).copy()
    model = smf.ols("CAR_short ~ firm_size_log * C(announcement_type)", data=model_df).fit()
    print(model.summary())

    X = sm.add_constant(pd.get_dummies(
        model_df[["firm_size_log", "announcement_type"]], drop_first=True
    ).astype(float))
    vif = pd.DataFrame({
        "variable": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    })
    print("\nVariance Inflation Factors:")
    print(vif)

    return {"RQ": "RQ3", "method": "multiple regression with interaction",
            "statistic": model.fvalue, "p_value": model.f_pvalue,
            "n": len(model_df), "r_squared": model.rsquared,
            "adj_r_squared": model.rsquared_adj}


rq3_result = rq3_test(df)

                            OLS Regression Results                            
Dep. Variable:              CAR_short   R-squared:                       0.590
Model:                            OLS   Adj. R-squared:                  0.334
Method:                 Least Squares   F-statistic:                     2.306
Date:                Sun, 26 Jul 2026   Prob (F-statistic):              0.141
Time:                        15:20:33   Log-Likelihood:                 35.019
No. Observations:                  14   AIC:                            -58.04
Df Residuals:                       8   BIC:                            -54.20
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                                        coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------

## RQ4 — Does CAR differ between tech and non-tech sectors?

In [ ]:
def rq4_test(df):
    tech = df[df["sector_binary"] == "Technology"]["CAR_short"].dropna()
    nontech = df[df["sector_binary"] == "Non-Technology"]["CAR_short"].dropna()

    _, p_levene = stats.levene(tech, nontech)
    equal_var = p_levene >= ALPHA
    variance_note = "equal variance assumed" if equal_var else "Welch's correction applied"
    print(f"Levene's test (equal variance): p = {p_levene:.4f} -> {variance_note}")

    t_stat, p_val = stats.ttest_ind(tech, nontech, equal_var=equal_var)
    mean_diff = tech.mean() - nontech.mean()
    print(f"Independent-samples t-test: t = {t_stat:.3f}, p = {p_val:.4f}")
    print(f"Mean CAR difference (tech - non-tech): {mean_diff:.4%}")
    print(f"Result: {'REJECT' if p_val < ALPHA else 'FAIL TO REJECT'} H0 at alpha = .05")

    return {"RQ": "RQ4", "method": "independent-samples t-test", "statistic": t_stat,
            "p_value": p_val, "n": len(tech) + len(nontech), "mean_diff": mean_diff}


rq4_result = rq4_test(df)

Levene's test (equal variance): p = 0.7054 -> equal variance assumed
Independent-samples t-test: t = 0.006, p = 0.9951
Mean CAR difference (tech - non-tech): 0.2016%
Result: FAIL TO REJECT H0 at alpha = .05


## Save all results

In [ ]:
results = [rq1_result, rq2_result, rq3_result, rq4_result]
results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(RESULTS_DIR, "rq_results_summary.csv"), index=False)
print(f"All results saved -> {RESULTS_DIR}/rq_results_summary.csv")
results_df

All results saved -> /content/QM640-WALSH-CAPSTONE/results/rq_results_summary.csv


,RQ,method,statistic,p_value,n,CAAR,eta_squared,r_squared,adj_r_squared,mean_diff
0,RQ1,Wilcoxon signed-rank (non-parametric fallback),54297.000000,0.001408,509,-0.024583,NaN,NaN,NaN,NaN
1,RQ2,Kruskal-Wallis (non-parametric fallback),3.877862,0.143858,509,NaN,0.00665,NaN,NaN,NaN
2,RQ3,multiple regression with interaction,2.306138,0.140540,14,NaN,NaN,0.590388,0.334381,NaN
3,RQ4,independent-samples t-test,0.006157,0.995090,509,NaN,NaN,NaN,NaN,0.002016


## Commit and push results back to GitHub

In [ ]:
!git -C {BASE_DIR} add "results/rq_results_summary.csv"
!git -C {BASE_DIR} commit -m "Step 6: RQ1-RQ4 statistical test results"
!git -C {BASE_DIR} push

[main a75cdd5] Step 6: RQ1-RQ4 statistical test results
 1 file changed, 5 insertions(+), 5 deletions(-)
 rewrite results/rq_results_summary.csv (82%)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 693 bytes | 693.00 KiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   71873a6..a75cdd5  main -> main
